In [4]:
import pandas as pd
import torch 
import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import math

In [6]:
df = pd.read_json("rawInput.json")

SEPARATOR = "---------------------"

# process each poem into formatted string
def format_poem(row):
    author = row["Author"]
    title = row["Title"]
    text = row["text"]
    return f"{author}\n{title}:\n\n{text}\n\n{SEPARATOR}"

# combine all poems
output_text = "\n".join(df.apply(format_poem, axis=1))

# write to input.txt
with open("input.txt", "w", encoding="utf-8") as f:
    f.write(output_text)

print(f"Processed {len(df)} poems to input.txt")
print(f"Total characters: {len(output_text)}")


Processed 38499 poems to input.txt
Total characters: 90332083


In [7]:
chars = sorted(list(set(output_text))) # unique characters in the text
vocab_size = len(chars)

stoi = { ch:i for i,ch in enumerate(chars) } # char to index
itos = { i:ch for i,ch in enumerate(chars) } # index to char

encode = lambda s: [stoi[c] for c in s] # string to list of ints
decode = lambda l: ''.join([itos[i] for i in l]) # list of ints to string

print(chars)
print(len(chars))

['\x01', '\x05', '\x07', '\t', '\n', '\x0e', '\x16', '\x1a', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\xa0', 'æ', 'ë', '–', '—', '‘', '’']
110


In [8]:
print(encode("hello!"))
print(decode(encode("hello!")))


[80, 77, 84, 84, 87, 9]
hello!


In [23]:
# ----- hyperparameters ------
batch_size = 15 # sequences to be processed in parallel
sequence_size = 100 # maximum context size
max_iterations = 2500 
heads = 25 # arbitrary
n_layers = 6 # number of layers
d_model = 250 
dropout = 0.25 # toggle for dropout
learning_rate = 1e-3 
device = 'cuda' if torch.cuda.is_available() else 'cpu' # check for gpu availability
d_k = d_v = d_model // heads  # dimensions for the heads assuming equi-split (integer division)

# ----------------------------- 

# worth noting this is just a generation task so a decoder will suffice - no encoders in this code
# lets first get the data prepared

data = torch.tensor(encode(output_text), dtype=torch.long)
n = int(0.9*len(data)) # use 90% of the text as training and 10% for validation
train_data = data[:n]
val_data = data[n:]

# ok now load the data

def get_batch(operation): # returns inputs x and the target outputs y we'll need for loss calculation
    data = train_data if operation == 'train' else val_data
    ix = torch.randint(len(data)-sequence_size, (batch_size,))
    x = torch.stack([data[i:i+sequence_size] for i in ix])
    y = torch.stack([data[i+1:i+sequence_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x,y # x here is just a 2D array of form ([a,b,c,..], ..)

class Embedding(nn.Module):
    def __init__(self): # embedding network with learned embeddings
        super().__init__()
        self.positional_embedding = nn.Embedding(sequence_size, d_model)
        self.token_embedding = nn.Embedding(vocab_size, d_model) # keep in mind the integers get upscaled to be R^d_model
    
    def forward(self, xs): # in transformers, you just add positional and token embeddings
        B, T = xs.shape
        positions = torch.arange(T, device=xs.device)  # position indices: 0, 1, 2, ..., T-1
        return self.token_embedding(xs) + self.positional_embedding(positions)

 
class FeedForward(nn.Module): # intermediate ff network - two linear activations with relu inbetween
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model), # weird intermediary expansion by 4 in paper
            nn.Dropout(dropout) # optional for dropout
        )
    
    def forward(self, x):
        return self.net(x) # just return the output
    
class AttentionHead(nn.Module): # attention head
    def __init__(self, d_model):
        super().__init__()
        self.w_q = nn.Linear(d_model, d_k, bias=False)
        self.w_k = nn.Linear(d_model, d_k, bias=False)
        self.w_v = nn.Linear(d_model, d_k, bias=False)

        self.register_buffer('tril', torch.tril(torch.ones(sequence_size, sequence_size)))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, isMasked=False): # renamed from self_attention_forward
        B, T, C = x.shape

        Q = self.w_q(x) # multiplied by the learned subspace maps
        K = self.w_k(x)
        V = self.w_v(x)

        attn = (Q @ K.transpose(-2,-1)) / math.sqrt(d_k)
        if isMasked: # decoder block needs masking
            attn = attn.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        return attn @ V 

class MultiAttentionHead(nn.Module):
    def __init__(self, n_heads, d_model, isMasked=False): # might as well keep optionality here
        super().__init__()
        self.isMasked = isMasked
        self.heads = nn.ModuleList([AttentionHead(d_model) for _ in range(n_heads)])
        self.proj_map = nn.Linear(d_model, d_model) # W^O map to take the concat. heads back to their original space
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        output = torch.cat([h(x, isMasked=self.isMasked) for h in self.heads], dim=-1)
        output = self.dropout(self.proj_map(output))
        return output
    
class Block(nn.Module):  # single block definition - 1 attention head and 1 feedforward
    def __init__(self):
        super().__init__()

        self.sa = MultiAttentionHead(heads, d_model, isMasked=True) # define the MHA and FF operations
        self.ffwd = FeedForward(d_model)

        self.ln1 = nn.LayerNorm(d_model) # ok now layer normals
        self.ln2 = nn.LayerNorm(d_model) # second one for after output

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # skip implementation to allow identity learning
        x = x + self.ffwd(self.ln2(x)) 
        return x 
        

class Transformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = Embedding()
        self.blocks = nn.Sequential(*[Block() for _ in range(n_layers)])
        self.finalLayerNorm = nn.LayerNorm(d_model) # final layer norm from last block
        self.fLayer = nn.Linear(d_model, vocab_size) # final linear layer output (softmax should be applied after)
    
    def forward(self, xs, targets=None):
        B, T = xs.shape # batch size and sequence length
        xps = self.embedding(xs) # map inputs to embeddings
        xps = self.blocks(xps)
        xps = self.finalLayerNorm(xps)
        logits = self.fLayer(xps)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B*T, C)
            targets_flat = targets.view(B*T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    def generate(self, xs, max_new_tokens):
        for _ in range(max_new_tokens):
            xs_cropped = xs[:, -sequence_size:]  # stay within context window
            logits, _ = self.forward(xs_cropped)
            logits = logits[:, -1, :]  # last position only (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            next_char = torch.multinomial(probs, num_samples=1)  # sample from distribution
            xs = torch.cat([xs, next_char], dim=1)
        return xs

In [24]:
# Initialize model and optimizer
model = Transformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Print number of parameters
print(f"Model has {sum(p.numel() for p in model.parameters())/1e6:.2f}M parameters")

@torch.no_grad()
def estimate_loss(eval_iters=200):
    """Estimate loss on train and val sets"""
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Training loop
eval_interval = 1000
for iter in range(max_iterations):
    
    # Evaluate periodically
    if iter % eval_interval == 0 or iter == max_iterations - 1:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    
    # Get batch and compute loss
    x, y = get_batch('train')
    logits, loss = model(x, y)
    
    # Backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training complete!")

# Generate some text
print("\n--- Generated Text ---")
context = torch.zeros((1, 1), dtype=torch.long, device=device)  # start with a single token (0)
generated = model.generate(context, max_new_tokens=1000)
print(decode(generated[0].tolist()))

Model has 2.34M parameters
Step 0: train loss 4.9321, val loss 4.9339
Step 1000: train loss 1.9872, val loss 1.9838
Step 2000: train loss 1.8471, val loss 1.8402
Step 2499: train loss 1.8123, val loss 1.8103
Training complete!

--- Generated Text ---
ng live.
When he watchors drug warsand vine.
But at weet the good I see: As or the foe, Arrit,
Nand foath alwar.
And loves faces, - Eaching bosser's Sake of Dow, say -
Till stong it telke,
Sowers the agon, dean's for alow,
'But!" and say, and freet, wildutinate Obhit,
Of biren's, vieriman truents
This sofe in now and trine? pers and as keed at and the sows grest makes, sadey neven, love outial sgainon the crivent fetaka row
The sping swumulder an feers,
Love looved too, ore benind hoyw, fut dield me all as so we contrake soll to thed nathe,
It tress is neved on, and himrown.
Wer be whatch doen the seellin!
But soone foll evern'  Arer, whing flarining in diembroce
Parnal and brusside of held fatell,
By for wentespoging 
Tempodids cried, no